# DeepCpx-CNN-Right — Complex STFT direction (Colab)

Trains the remaining **deep complex-STFT** expert (`deep_cpx_2class_right_100ep`) on GPU.

| Artifact | Role |
|----------|------|
| `best.pt` | Best validation-loss weights |
| `last.pt` | Full resume state (model + optim + history + mid-epoch batch) |
| `train_summary.json` / `best_history.json` / `run_config.json` | Run metadata |

**Resume after disconnect**
- End of epoch → `last.pt` (`batch_idx=0`)
- Every `CHECKPOINT_EVERY_BATCHES` train batches → mid-epoch `last.pt`
- Re-run all cells with `AUTO_RESUME=True`; training continues from Drive

**Drive layout (this model)**
```
.../DopplerLab/cpx/
  notebooks/          # keep a copy of this notebook here
  checkpoints/transfer/direction/<RUN_NAME>/
  outputs/transfer/direction/<RUN_NAME>/
```

**Data** — this is IDMT direction training (not VS13 diffusion).
Put **IDMT-Traffic** on Drive with `audio/` + `annotation/` (default path in the config cell).

**Recipe** — Phase B deep CNN, `feature_type=complex_stft`, mono=`right`, 2-class,
early-stop after **min 20 epochs** (`preempt`, patience 15).

## 1. Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q librosa soundfile scikit-learn tqdm pandas torch

## 2. Configuration (edit paths here)

In [ ]:
from pathlib import Path

# Shared DopplerLab root on Shared drives
SHARED_ROOT = Path(
    '/content/drive/Shareddrives/Spectral Transformers - Doppler/DopplerLab'
)

# All notebooks / checkpoints / outputs for this complex-STFT track
CPX_ROOT = SHARED_ROOT / 'cpx'

# Parent datasets folder (VS13 lives here for diffusion; IDMT is separate)
DATASET_ROOT = SHARED_ROOT / 'datasets' / 'vs13'  # as documented; IDMT path below

# IDMT-Traffic root (must contain audio/ + annotation/) — edit if yours differs
DATA_DIR = SHARED_ROOT / 'datasets' / 'IDMT_Traffic'
# Fallbacks if you keep IDMT next to the old MyDrive layout or under cpx:
for _cand in [
    DATA_DIR,
    SHARED_ROOT / 'IDMT_Traffic',
    CPX_ROOT / 'IDMT_Traffic',
    Path('/content/drive/MyDrive/DopplerLab/IDMT_Traffic'),
]:
    if (_cand / 'audio').is_dir():
        DATA_DIR = _cand
        break

CHECKPOINT_DIR = CPX_ROOT / 'checkpoints'
OUTPUT_DIR = CPX_ROOT / 'outputs'
NOTEBOOKS_DIR = CPX_ROOT / 'notebooks'

# --- Training (remaining model: deep cpx RIGHT) ---
RUN_NAME = 'deep_cpx_2class_right_100ep'
FEATURE_TYPE = 'complex_stft'
MONO_SOURCE = 'right'          # 'mean' | 'left' | 'right'
EPOCHS = 100
BATCH_SIZE = 8                 # complex STFT is wide; lower if CUDA OOM
PREEMPT = True                 # early stop after MIN_EPOCHS
MIN_EPOCHS = 20                # never stop before this epoch
PATIENCE = 15
CHECKPOINT_EVERY_BATCHES = 25  # mid-epoch Drive resume granularity
AUTO_RESUME = True
FRESH_TRAINING = False         # True = ignore last.pt once (new run)

# --- Code package (idmt_experiments) ---
# Easiest: upload ONLY the folder
#   DopplerLab/IDMT_experiments/src
# to Drive as:
#   .../cpx/IDMT_experiments/src
# (must contain pyproject.toml + idmt_experiments/)
#
# Or set PKG_SRC to wherever you put that folder:
PKG_SRC = CPX_ROOT / 'IDMT_experiments' / 'src'   # <-- preferred upload path
REPO_DIR = CPX_ROOT / 'DopplerLab'                 # optional full-repo clone
USE_GIT_CLONE = False
GIT_URL = 'https://github.com/YOUR_ORG/DopplerLab.git'

RUN_DIR = CHECKPOINT_DIR / 'transfer' / 'direction' / RUN_NAME
for d in (RUN_DIR, OUTPUT_DIR, NOTEBOOKS_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('CPX_ROOT     :', CPX_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('RUN_DIR      :', RUN_DIR)
print('OUTPUT_DIR   :', OUTPUT_DIR)

## 3. Install `idmt_experiments`

In [ ]:
import sys
import subprocess
from pathlib import Path

if USE_GIT_CLONE and not (REPO_DIR / 'IDMT_experiments').exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['git', 'clone', '--depth', '1', GIT_URL, str(REPO_DIR)])

candidates = [
    PKG_SRC,
    CPX_ROOT / 'IDMT_experiments' / 'src',
    CPX_ROOT / 'src',                          # if you uploaded just `src` as cpx/src
    REPO_DIR / 'IDMT_experiments' / 'src',
    SHARED_ROOT / 'DopplerLab' / 'IDMT_experiments' / 'src',
    Path('/content/DopplerLab/IDMT_experiments/src'),
    Path('/content/IDMT_experiments/src'),
    Path('/content/src'),
]
SRC = None
for c in candidates:
    if (c / 'idmt_experiments').is_dir() and (c / 'pyproject.toml').is_file():
        SRC = c
        break
if SRC is None:
    raise FileNotFoundError(
        'Cannot find package src (needs pyproject.toml + idmt_experiments/). '
        f'Upload IDMT_experiments/src to {CPX_ROOT / "IDMT_experiments" / "src"} '
        'or set PKG_SRC in the config cell. Checked:\\n  ' + '\\n  '.join(str(c) for c in candidates)
    )

# pyproject.toml lives in IDMT_experiments/src
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SRC)])
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Using package src:', SRC)

In [ ]:
import idmt_experiments.config as cfg_mod
from idmt_experiments.config import DirectionConfig
from idmt_experiments.cnn.train_recipe import phase_a_recipe
from idmt_experiments.engine.train_mel import train_mel_split
from idmt_experiments.transfer.eval import run_eval
from idmt_experiments.transfer.model import build_model
from idmt_experiments.training_resume import last_state_path, load_training_state
from idmt_experiments.src.preprocess import resolve_data_dir

cfg_mod.DEFAULT_CHECKPOINT_DIR = CHECKPOINT_DIR
cfg_mod.DEFAULT_OUTPUT_DIR = OUTPUT_DIR
cfg_mod.DEFAULT_DATA_DIR = DATA_DIR

data_root = resolve_data_dir(DATA_DIR)
assert (data_root / 'audio').is_dir(), f'Missing audio/: {data_root}'
assert (data_root / 'annotation').is_dir(), f'Missing annotation/: {data_root}'
print('Data OK:', data_root)

## 4. Resume status (read before training)

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

ckpt_path = RUN_DIR / 'best.pt'
last_path = last_state_path(ckpt_path)

if FRESH_TRAINING and last_path.exists():
    print('FRESH_TRAINING=True — ignoring existing last.pt for this run')
    resume = False
else:
    resume = AUTO_RESUME and last_path.exists()

if resume:
    st = load_training_state(ckpt_path, device)
    if st.mid_epoch:
        print(f'Will resume MID-EPOCH {st.epoch}, batch {st.batch_idx}')
    else:
        print(f'Will resume after completed epoch {st.epoch} -> start {st.epoch + 1}')
    print(f'  best_epoch={st.best_epoch}  best_val_acc={st.best_val_acc:.4f}')
    print(f'  history_len={len(st.history)}  last.pt={last_path}')
else:
    print('Fresh training (no last.pt on Drive, or FRESH_TRAINING=True)')
    print('  run dir:', RUN_DIR)

## 5. Train DeepCpx-CNN-Right (`complex_stft`)

In [ ]:
cfg = DirectionConfig(
    task='direction',
    feature_type=FEATURE_TYPE,
    mono_source=MONO_SOURCE,
    n_classes=2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    preempt=PREEMPT,
    min_epochs=MIN_EPOCHS,
    patience=PATIENCE,
    weight_decay=1e-3,
)
recipe = phase_a_recipe()

print('=' * 72)
print('DeepCpx-CNN — Phase B transfer')
print(f'  run         : {RUN_NAME}')
print(f'  feature     : {cfg.feature_type}')
print(f'  mono        : {cfg.mono_source}')
print(f'  epochs      : {cfg.epochs}  batch={cfg.batch_size}')
print(f'  preempt     : {cfg.preempt}  min_epochs={cfg.min_epochs}  patience={cfg.patience}')
print(f'  resume      : {resume}')
print(f'  checkpoints : {CHECKPOINT_DIR}')
print('=' * 72)

ckpt = train_mel_split(
    run_name=RUN_NAME,
    cfg=cfg,
    model_factory=lambda c: build_model(c, 'deep_mel_cnn'),
    checkpoint_subdir='transfer/direction',
    checkpoint_dir=CHECKPOINT_DIR,
    data_dir=DATA_DIR,
    device=device,
    resume=resume,
    train_recipe=recipe,
    extra_ckpt={
        'backbone': 'deep_mel_cnn',
        'phase': 'B',
        'colab': True,
        'cpx_root': str(CPX_ROOT),
    },
    checkpoint_every_batches=CHECKPOINT_EVERY_BATCHES,
)
print('Best checkpoint:', ckpt)

## 6. Training history

In [ ]:
import json
import matplotlib.pyplot as plt

hist_path = RUN_DIR / 'best_history.json'
sum_path = RUN_DIR / 'train_summary.json'
if not hist_path.exists() and sum_path.exists():
    hist = json.loads(sum_path.read_text())['history']
else:
    hist = json.loads(hist_path.read_text()) if hist_path.exists() else []

if not hist:
    print('No history yet')
else:
    epochs = [h['epoch'] for h in hist]
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, [h['train_acc'] for h in hist], label='train_acc')
    plt.plot(epochs, [h['val_acc'] for h in hist], label='val_acc')
    if 'val_bal_acc' in hist[0]:
        plt.plot(epochs, [h['val_bal_acc'] for h in hist], label='val_bal_acc')
    plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.title(RUN_NAME)
    fig_path = OUTPUT_DIR / 'transfer' / 'direction' / RUN_NAME / 'history.png'
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved', fig_path)
    if sum_path.exists():
        print(sum_path.read_text()[:500])

## 7. Evaluate on EUSIPCO test

In [ ]:
import idmt_experiments.config as cfg_mod

cfg_mod.DEFAULT_OUTPUT_DIR = OUTPUT_DIR
metrics = run_eval(
    ckpt,
    data_dir=DATA_DIR,
    output_subdir='transfer/direction',
    device=device,
)
print(metrics)

## 8. Artifacts on Drive

Already under `CPX_ROOT`:

- `cpx/checkpoints/transfer/direction/deep_cpx_2class_right_100ep/`
- `cpx/outputs/transfer/direction/deep_cpx_2class_right_100ep/`

Copy `best.pt` / `last.pt` / summaries back to your PC under
`IDMT_experiments/checkpoints/transfer/direction/<RUN_NAME>/` when ready to fuse with left.

In [ ]:
from google.colab import files

DOWNLOAD = False  # set True to pull files to your browser download folder
if DOWNLOAD:
    for name in ['best.pt', 'last.pt', 'train_summary.json', 'best_history.json', 'run_config.json']:
        p = RUN_DIR / name
        if p.exists():
            files.download(str(p))
            print('downloaded', p.name)
        else:
            print('missing', p)
else:
    print('Artifacts stay on Drive at', RUN_DIR)

## Colab tips

1. **Runtime disconnected?** Re-run from section 1. `AUTO_RESUME=True` picks up `last.pt` on Drive (incl. mid-epoch).
2. **CUDA OOM?** Lower `BATCH_SIZE` (try 4) and re-run; resume still works.
3. **Start over:** set `FRESH_TRAINING=True` once, or delete the run folder on Drive / change `RUN_NAME`.
4. **Extend training:** raise `EPOCHS` and re-run section 5.
5. Keep a copy of this notebook under `cpx/notebooks/` so the Shared drive is self-contained.
6. After right finishes, fuse with local/Drive left checkpoint:
   `python -m idmt_experiments.fusion --left-checkpoint .../deep_cpx_2class_left_100ep/best.pt --right-checkpoint .../deep_cpx_2class_right_100ep/best.pt --run-name fusion_cpx_2class_100ep`